# ML Pipeline for 5Ghz Device Prediction Positioning Model

This notebook contains the machine learning pipeline used to train a device positioning model from the labelled dataset in `ml_dataset.parquet`.

<br>

---

<br>

## Regression Problem

The machine learning task is formulated as a supervised multi-output regression problem:

$$
f(\mathbf{x}_i) \to \mathbf{y}_i =
\begin{bmatrix}
x_i \\
y_i
\end{bmatrix}
$$

where:

- $\mathbf{x}_i$ is the input feature vector for the $i$-th sample
- $\mathbf{y}_i$ is the ground-truth 2D target position
- $f(\cdot)$ is the learned regression function

Equivalently, the model learns:

$$
\hat{\mathbf{y}}_i = f(\mathbf{x}_i)
$$

with prediction error measured by the Euclidean localisation error:

$$
e_i = \sqrt{(\hat{x}_i - x_i)^2 + (\hat{y}_i - y_i)^2}
$$

The training objective is to learn a function $f$ that minimises prediction error over all training samples.

This notebook follows four core principles:
1. Use one row per `scenario_id` + `target_id`.
2. Split by unseen `scenario_id` groups rather than by random rows.
3. Compare against conventional RSSI/TDOA/DOA baselines without using them as ML inputs.
4. Enforce a direct telemetry-to-coordinate learning setup with leakage checks.

<br>

---

<br>

## Data Split Strategy

To evaluate generalisation to unseen environments, the dataset **(100 network environments)** is split by unique `scenario_id` groups. This prevents the model from seeing the same environment in both training and evaluation, which would otherwise inflate performance.

The grouped split used in this notebook is:

- **Training set (70%)**  = 70 environments
- **Validation set (20%)**  = 20 environments
- **Testing set (10%)**  = 10 environments

<br>

#### Note:
> Because the split is group-based, the exact environment counts must be rounded to whole numbers.
> One important constraint is that no `scenario_id` appears in more than one split.


In [ ]:
%pip install -q pandas pyarrow scikit-learn matplotlib seaborn

In [ ]:
import gc
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 13  # Keeping it the same as the environment seed for reproducibility

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

IDENTIFIER_COLUMNS = ["scenario_id", "target_id"]
TARGET_COLUMNS = ["target_x", "target_y"]
GENERATED_CATEGORY_CODE_SUFFIX = "_category_code"

FORBIDDEN_EXACT_FEATURE_COLUMNS = {
    "seed",
    "target_label",
    "link_count",
    "rssi_anchor_count",
    "tdoa_anchor_count",
    "doa_anchor_count",
}
FORBIDDEN_PREFIXES = ("target_", "rssi_est", "tdoa_est", "doa_est", "link_", "true_")
FORBIDDEN_SUFFIXES = ("_est_x", "_est_y", "_error_m", "_residual_rmse_m", "_success", "_env_type")
FORBIDDEN_TOKENS = (
    "distance",
    "path_loss",
    "attenuation",
    "blocker",
    "loss",
    "noise",
    "sigma",
    "ideal",
    "arrival_time",
    "link_state",
)


def is_blocked_feature(column):
    if column in IDENTIFIER_COLUMNS or column in TARGET_COLUMNS:
        return True
    return (
        column in FORBIDDEN_EXACT_FEATURE_COLUMNS
        or column.startswith(FORBIDDEN_PREFIXES)
        or column.endswith(FORBIDDEN_SUFFIXES)
        or any(token in column for token in FORBIDDEN_TOKENS)
    )


def read_parquet_columns_downcast(path, columns, identifier_columns=()):
    table = pq.read_table(path, columns=columns)
    identifier_set = set(identifier_columns)
    cast_fields = []

    for field in table.schema:
        if field.name in identifier_set:
            cast_fields.append(field)
        elif pa.types.is_floating(field.type):
            cast_fields.append(pa.field(field.name, pa.float32()))
        elif pa.types.is_signed_integer(field.type):
            cast_fields.append(pa.field(field.name, pa.int32()))
        elif pa.types.is_unsigned_integer(field.type):
            cast_fields.append(pa.field(field.name, pa.uint32()))
        else:
            cast_fields.append(field)

    table = table.cast(pa.schema(cast_fields), safe=False)
    try:
        frame = table.to_pandas(split_blocks=True, self_destruct=True)
    except TypeError:
        frame = table.to_pandas(split_blocks=True)
    del table
    gc.collect()
    return frame

def localisation_error_m(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.sum((y_true - y_pred) ** 2, axis=1))


def metric_row(split_name, model_name, y_true, y_pred):
    errors = localisation_error_m(y_true, y_pred)
    return {
        "split": split_name,
        "model": model_name,
        "sample_count": int(len(errors)),
        "mean_error_m": float(errors.mean()),
        "median_error_m": float(np.median(errors)),
        "rmse_m": float(np.sqrt(np.mean(errors ** 2))),
        "p90_error_m": float(np.quantile(errors, 0.90)),
    }


def describe_split(name, frame):
    print(
        f"{name}: rows={len(frame):,}, scenarios={frame['scenario_id'].nunique()}"
    )


## 1. Load `ml_dataset.parquet` and Baseline Estimates

In [ ]:
# Set DATA_PATH to the local leakage-safe ML dataset file.
DATA_PATH = Path("/ML Pipeline/ml_dataset.parquet").resolve()

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Update DATA_PATH so it points to a local ml_dataset.parquet file."
    )

DATA_DIR = DATA_PATH.parent
POSITION_ESTIMATES_PATH = DATA_DIR / "position_estimates.parquet"
if not POSITION_ESTIMATES_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {POSITION_ESTIMATES_PATH}. Conventional baselines are loaded separately from position_estimates.parquet."
    )

dataset_columns = list(pq.read_schema(DATA_PATH).names)
position_estimate_columns = list(pq.read_schema(POSITION_ESTIMATES_PATH).names)

required_ml_columns = IDENTIFIER_COLUMNS + TARGET_COLUMNS
missing_ml_columns = [column for column in required_ml_columns if column not in dataset_columns]
if missing_ml_columns:
    raise ValueError(f"ml_dataset.parquet is missing required columns: {missing_ml_columns}")

raw_feature_columns = [
    column
    for column in dataset_columns
    if not is_blocked_feature(column) and not column.endswith(GENERATED_CATEGORY_CODE_SUFFIX)
]
excluded_columns = sorted(column for column in dataset_columns if is_blocked_feature(column))
if not raw_feature_columns:
    raise ValueError("No leakage-safe feature columns were found in ml_dataset.parquet.")

required_estimate_columns = [
    "scenario_id",
    "target_id",
    "target_x",
    "target_y",
    "rssi_est_x",
    "rssi_est_y",
    "tdoa_est_x",
    "tdoa_est_y",
    "doa_est_x",
    "doa_est_y",
]
missing_estimate_columns = [
    column for column in required_estimate_columns if column not in position_estimate_columns
]
if missing_estimate_columns:
    raise ValueError(
        f"position_estimates.parquet is missing required baseline columns: {missing_estimate_columns}"
    )

dataset_df = read_parquet_columns_downcast(
    DATA_PATH,
    required_ml_columns + raw_feature_columns,
    identifier_columns=IDENTIFIER_COLUMNS,
)
position_estimates_df = read_parquet_columns_downcast(
    POSITION_ESTIMATES_PATH,
    required_estimate_columns,
    identifier_columns=IDENTIFIER_COLUMNS,
)

label_check_df = dataset_df[["scenario_id", "target_id", "target_x", "target_y"]].merge(
    position_estimates_df[["scenario_id", "target_id", "target_x", "target_y"]],
    on=["scenario_id", "target_id"],
    how="inner",
    suffixes=("", "_estimate"),
    validate="one_to_one",
)
if len(label_check_df) != len(dataset_df):
    raise ValueError("position_estimates.parquet does not cover every ML dataset scenario-target row.")
for target_column in ["target_x", "target_y"]:
    estimate_column = f"{target_column}_estimate"
    if not np.allclose(
        label_check_df[target_column].astype(float),
        label_check_df[estimate_column].astype(float),
        equal_nan=False,
    ):
        raise ValueError(f"{target_column} labels differ between ml_dataset.parquet and position_estimates.parquet.")

print(f"Loaded ML dataset: {DATA_PATH}")
print(f"Loaded conventional baselines: {POSITION_ESTIMATES_PATH}")
print(f"Rows: {len(dataset_df):,}")
print(f"Unique scenarios: {dataset_df['scenario_id'].nunique()}")
print(f"ML columns read: {len(dataset_df.columns):,} of {len(dataset_columns):,}")
display(dataset_df.head())


In [ ]:
baseline_error_cols = [
    column for column in ["rssi_error_m", "tdoa_error_m", "doa_error_m"]
    if column in position_estimates_df.columns
]

if baseline_error_cols:
    baseline_summary_df = (
        position_estimates_df[baseline_error_cols]
        .describe()
        .T[["mean", "50%", "max"]]
        .rename(columns={"50%": "median"})
    )
    display(baseline_summary_df)

scenario_count = dataset_df["scenario_id"].nunique()
if scenario_count < 10:
    warnings.warn(
        "This dataset has fewer than 10 scenarios. The notebook will run, but should probably use the full 100-scenario dataset for the real experiment.",
        stacklevel=2,
    )


## 2. Keep only leakage-safe dynamic features

The model feature matrix is selected dynamically from `ml_dataset.parquet` using a blocklist. This keeps new per-antenna and per-pair observed telemetry features without admitting target labels, conventional solver outputs, true geometry, path-loss internals, attenuation decomposition, or noise parameters.


In [ ]:
if "raw_feature_columns" not in globals():
    source_columns_for_filtering = list(dataset_df.columns)
    raw_feature_columns = [
        column
        for column in source_columns_for_filtering
        if not is_blocked_feature(column) and not column.endswith(GENERATED_CATEGORY_CODE_SUFFIX)
    ]
    excluded_columns = sorted(
        column for column in source_columns_for_filtering if is_blocked_feature(column)
    )

raw_feature_columns = [
    column
    for column in raw_feature_columns
    if column in dataset_df.columns and not column.endswith(GENERATED_CATEGORY_CODE_SUFFIX)
]

if not raw_feature_columns:
    raise ValueError("No leakage-safe feature columns were found in ml_dataset.parquet.")

categorical_source_columns = [
    column for column in raw_feature_columns if not pd.api.types.is_numeric_dtype(dataset_df[column])
]
available_feature_columns = raw_feature_columns.copy()
categorical_feature_encodings = {}

for column in categorical_source_columns:
    encoded_column = f"{column}{GENERATED_CATEGORY_CODE_SUFFIX}"
    values = dataset_df[column].astype("string")
    categories = sorted(values.dropna().unique())
    category_map = {value: index for index, value in enumerate(categories)}
    dataset_df[encoded_column] = values.map(category_map).astype("float32")
    available_feature_columns = [
        encoded_column if feature_column == column else feature_column
        for feature_column in available_feature_columns
    ]
    categorical_feature_encodings[column] = {
        "encoded_column": encoded_column,
        "category_count": len(categories),
    }

model_df = dataset_df

wide_feature_counts = {
    "antenna_layout": sum(column.startswith("antenna_") and column != "antenna_count" for column in available_feature_columns),
    "rssi_per_antenna": sum(column.startswith("rssi_antenna_") for column in available_feature_columns),
    "doa_per_antenna": sum(column.startswith("doa_antenna_") for column in available_feature_columns),
    "tdoa_per_pair": sum(column.startswith("tdoa_ref_") for column in available_feature_columns),
}

assert "scenario_id" not in available_feature_columns
assert "target_x" not in available_feature_columns
assert "target_y" not in available_feature_columns
assert not any(column.startswith(("rssi_est", "tdoa_est", "doa_est")) for column in available_feature_columns)
assert not any(column.endswith(FORBIDDEN_SUFFIXES) for column in available_feature_columns)
assert not any(column.startswith("link_") for column in available_feature_columns)
assert not any(any(token in column for token in FORBIDDEN_TOKENS) for column in available_feature_columns)

print(f"Features kept: {len(available_feature_columns)}")
print(f"Columns intentionally excluded: {len(excluded_columns)}")
print("\nWide telemetry feature counts:")
print(wide_feature_counts)
print("\nCategorical feature encodings:")
print(categorical_feature_encodings)
print("\nKept feature columns:")
print(available_feature_columns)
print("\nExamples of excluded leakage or simulator-only columns:")
print(excluded_columns[:30])

display(model_df[IDENTIFIER_COLUMNS + TARGET_COLUMNS + available_feature_columns].head())


## 3. Split by unseen scenarios

Never random-split rows for the main experiment. The training set, validation set, and test set must use different `scenario_id` groups. This notebook uses a grouped 70:20:10 split.


In [ ]:
TRAIN_SIZE = 0.70
VALID_SIZE = 0.20
TEST_SIZE = 0.10


def grouped_train_valid_test_split_indices(
    frame,
    group_col="scenario_id",
    train_size=TRAIN_SIZE,
    valid_size=VALID_SIZE,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
):
    total = train_size + valid_size + test_size
    if not np.isclose(total, 1.0):
        raise ValueError("train_size, valid_size, and test_size must sum to 1.0")

    unique_groups = pd.Series(frame[group_col].drop_duplicates().to_numpy())
    if len(unique_groups) < 3:
        raise ValueError("Need at least 3 unique scenarios for train/valid/test splitting.")

    outer_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state,
    )
    train_valid_group_idx, test_group_idx = next(
        outer_splitter.split(unique_groups, groups=unique_groups)
    )
    train_valid_groups = unique_groups.iloc[train_valid_group_idx]
    test_groups = set(unique_groups.iloc[test_group_idx])

    valid_fraction_of_remaining = valid_size / (train_size + valid_size)
    inner_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=valid_fraction_of_remaining,
        random_state=random_state + 1,
    )
    train_group_idx, valid_group_idx = next(
        inner_splitter.split(train_valid_groups, groups=train_valid_groups)
    )
    train_groups = set(train_valid_groups.iloc[train_group_idx])
    valid_groups = set(train_valid_groups.iloc[valid_group_idx])

    train_index = frame.index[frame[group_col].isin(train_groups)].to_numpy()
    valid_index = frame.index[frame[group_col].isin(valid_groups)].to_numpy()
    test_index = frame.index[frame[group_col].isin(test_groups)].to_numpy()
    return train_index, valid_index, test_index


train_index, valid_index, test_index = grouped_train_valid_test_split_indices(model_df)

train_df = model_df.loc[train_index, IDENTIFIER_COLUMNS + TARGET_COLUMNS]
valid_df = model_df.loc[valid_index, IDENTIFIER_COLUMNS + TARGET_COLUMNS]
test_df = model_df.loc[test_index, IDENTIFIER_COLUMNS + TARGET_COLUMNS]

describe_split("train", train_df)
describe_split("valid", valid_df)
describe_split("test", test_df)

print("\nScenario overlap checks")
print("train/valid overlap:", set(train_df["scenario_id"]) & set(valid_df["scenario_id"]))
print("train/test overlap:", set(train_df["scenario_id"]) & set(test_df["scenario_id"]))
print("valid/test overlap:", set(valid_df["scenario_id"]) & set(test_df["scenario_id"]))


## 4. Direct Conventional Baselines

Load RSSI, TDOA, and DOA/AOA conventional estimates only for held-out comparison. These estimate coordinates are not part of the ML feature matrix.


In [ ]:
BASELINE_COORD_COLUMNS = [
    "rssi_est_x",
    "rssi_est_y",
    "tdoa_est_x",
    "tdoa_est_y",
    "doa_est_x",
    "doa_est_y",
]

BASELINE_ESTIMATE_MAP = {
    "RSSI": ("rssi_est_x", "rssi_est_y"),
    "TDOA": ("tdoa_est_x", "tdoa_est_y"),
    "DOA": ("doa_est_x", "doa_est_y"),
}

baseline_estimates_df = position_estimates_df[
    IDENTIFIER_COLUMNS + BASELINE_COORD_COLUMNS
].drop_duplicates(IDENTIFIER_COLUMNS).copy()


def comparison_subset(row_index):
    comparison_df = model_df.loc[row_index, IDENTIFIER_COLUMNS + TARGET_COLUMNS].copy()
    comparison_df["_row_index"] = row_index
    comparison_df = comparison_df.merge(
        baseline_estimates_df,
        on=IDENTIFIER_COLUMNS,
        how="inner",
        validate="one_to_one",
    )
    finite_mask = np.isfinite(comparison_df[TARGET_COLUMNS].to_numpy(dtype=float)).all(axis=1)
    finite_mask &= np.isfinite(comparison_df[BASELINE_COORD_COLUMNS].to_numpy(dtype=float)).all(axis=1)
    comparison_df = comparison_df.loc[finite_mask].copy()
    comparison_index = comparison_df.pop("_row_index").to_numpy()
    return comparison_df, comparison_index


valid_eval_df, valid_eval_index = comparison_subset(valid_index)
test_eval_df, test_eval_index = comparison_subset(test_index)

y_train = model_df.loc[train_index, TARGET_COLUMNS].to_numpy(dtype=np.float32)
y_valid = valid_eval_df[TARGET_COLUMNS].to_numpy(dtype=np.float32)
y_test = test_eval_df[TARGET_COLUMNS].to_numpy(dtype=np.float32)

print(f"Validation comparison rows: {len(valid_eval_df):,}")
print(f"Test comparison rows: {len(test_eval_df):,}")

baseline_results = []
prediction_store = {}

for split_name, split_df in [("valid", valid_eval_df), ("test", test_eval_df)]:
    y_split = split_df[TARGET_COLUMNS].to_numpy(dtype=float)
    for label, columns in BASELINE_ESTIMATE_MAP.items():
        preds = split_df.loc[:, list(columns)].to_numpy(dtype=float)
        baseline_results.append(metric_row(split_name, label, y_split, preds))
        if split_name == "test":
            prediction_store[label] = preds

baseline_results_df = pd.DataFrame(baseline_results).sort_values(["split", "mean_error_m"])
display(baseline_results_df)


## 5. Direct Telemetry Models

Train supervised regressors from leakage-safe environment, antenna-layout, per-antenna, per-pair, and observed telemetry summaries to ground-truth target coordinates.


In [ ]:
RUN_MLP = False
TREE_MAX_TRAIN_ROWS = 250_000
MLP_MAX_TRAIN_ROWS = 100_000

numeric_feature_columns = [
    column
    for column in available_feature_columns
    if pd.api.types.is_numeric_dtype(model_df[column])
]
categorical_feature_columns = [
    column for column in available_feature_columns if column not in numeric_feature_columns
]

if categorical_feature_columns:
    raise ValueError(
        f"All ML feature columns should be numeric before training. Non-numeric columns: {categorical_feature_columns}"
    )

estimated_dense_train_gb = len(train_index) * len(available_feature_columns) * 8 / 1_000_000_000
print(f"Training rows: {len(train_index):,}")
print(f"Feature columns: {len(available_feature_columns):,}")
print(f"Dense float64 training matrix alone would be about {estimated_dense_train_gb:.2f} GB before model overhead.")


def capped_training_index(row_index, max_rows, label):
    if max_rows is None or len(row_index) <= max_rows:
        return row_index

    rng = np.random.default_rng(RANDOM_STATE)
    sampled_index = rng.choice(row_index, size=max_rows, replace=False)
    print(f"{label} training limited to {len(sampled_index):,} sampled rows to avoid RAM spikes.")
    return sampled_index


def make_tree_pipeline():
    return MultiOutputRegressor(
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_depth=6,
            max_iter=300,
            min_samples_leaf=32,
            random_state=RANDOM_STATE,
        )
    )


def make_mlp_pipeline():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                MLPRegressor(
                    hidden_layer_sizes=(128, 64),
                    activation="relu",
                    alpha=1e-4,
                    batch_size=1024,
                    max_iter=200,
                    early_stopping=True,
                    n_iter_no_change=15,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


tree_train_index = capped_training_index(train_index, TREE_MAX_TRAIN_ROWS, "Tree")

tree_pipeline = make_tree_pipeline()
tree_pipeline.fit(
    model_df.loc[tree_train_index, available_feature_columns],
    model_df.loc[tree_train_index, TARGET_COLUMNS].to_numpy(dtype=np.float32),
)

tree_valid_pred = tree_pipeline.predict(model_df.loc[valid_eval_index, available_feature_columns])
tree_test_pred = tree_pipeline.predict(model_df.loc[test_eval_index, available_feature_columns])

model_result_rows = [
    metric_row("valid", "Tree direct ML", y_valid, tree_valid_pred),
    metric_row("test", "Tree direct ML", y_test, tree_test_pred),
]
prediction_store["Tree direct ML"] = tree_test_pred

if RUN_MLP:
    mlp_train_index = capped_training_index(train_index, MLP_MAX_TRAIN_ROWS, "MLP")

    mlp_pipeline = make_mlp_pipeline()
    mlp_pipeline.fit(
        model_df.loc[mlp_train_index, available_feature_columns],
        model_df.loc[mlp_train_index, TARGET_COLUMNS].to_numpy(dtype=np.float32),
    )
    mlp_valid_pred = mlp_pipeline.predict(model_df.loc[valid_eval_index, available_feature_columns])
    mlp_test_pred = mlp_pipeline.predict(model_df.loc[test_eval_index, available_feature_columns])
    model_result_rows.extend(
        [
            metric_row("valid", "MLP direct ML", y_valid, mlp_valid_pred),
            metric_row("test", "MLP direct ML", y_test, mlp_test_pred),
        ]
    )
    prediction_store["MLP direct ML"] = mlp_test_pred
else:
    print("Skipping MLP direct ML by default because it densifies the wide feature matrix. Set RUN_MLP = True to run it on a capped sample.")

full_model_results_df = pd.DataFrame(model_result_rows).sort_values(["split", "mean_error_m"])

display(full_model_results_df)


In [ ]:
all_results_df = pd.concat(
    [baseline_results_df, full_model_results_df],
    ignore_index=True,
).sort_values(["split", "mean_error_m"])

display(all_results_df)
print("Primary v1 export model: Tree direct ML")


## 6. Refit the Primary Model and Export Predictions

After comparing models on the validation split, refit the primary tree model on `train + valid` and export predictions for every target in the held-out test scenarios.


In [ ]:
train_plus_valid_index = np.concatenate([train_index, valid_index])
primary_train_index = capped_training_index(
    train_plus_valid_index,
    TREE_MAX_TRAIN_ROWS,
    "Final tree",
)

primary_pipeline = make_tree_pipeline()
primary_pipeline.fit(
    model_df.loc[primary_train_index, available_feature_columns],
    model_df.loc[primary_train_index, TARGET_COLUMNS].to_numpy(dtype=np.float32),
)

final_test_predictions = primary_pipeline.predict(model_df.loc[test_eval_index, available_feature_columns])
final_test_predictions_all = primary_pipeline.predict(model_df.loc[test_index, available_feature_columns])

final_primary_results_df = pd.DataFrame(
    [metric_row("test", "Tree direct ML (refit train+valid)", y_test, final_test_predictions)]
)
display(final_primary_results_df)

predictions_df = model_df.loc[test_index, IDENTIFIER_COLUMNS].copy()
predictions_df["ml_est_x"] = final_test_predictions_all[:, 0]
predictions_df["ml_est_y"] = final_test_predictions_all[:, 1]

PREDICTIONS_PATH = DATA_DIR / "ml_predictions.parquet"
predictions_df.to_parquet(PREDICTIONS_PATH, index=False)

print(f"Saved predictions to: {PREDICTIONS_PATH}")
display(predictions_df.head())


In [ ]:
final_prediction_map = {
    "RSSI": prediction_store["RSSI"],
    "TDOA": prediction_store["TDOA"],
    "DOA": prediction_store["DOA"],
    "Tree direct ML": final_test_predictions,
}
if "MLP direct ML" in prediction_store:
    final_prediction_map["MLP direct ML"] = prediction_store["MLP direct ML"]


final_test_summary_df = pd.DataFrame(
    [metric_row("test", model_name, y_test, preds) for model_name, preds in final_prediction_map.items()]
).sort_values("mean_error_m")
display(final_test_summary_df)

plt.figure(figsize=(8, 5))
for model_name, preds in final_prediction_map.items():
    errors = np.sort(localisation_error_m(y_test, preds))
    cdf = np.arange(1, len(errors) + 1) / len(errors)
    plt.plot(errors, cdf, label=model_name)

plt.xlabel("Localisation error (m)")
plt.ylabel("CDF")
plt.title("Held-out test scenario comparison")
plt.legend()
plt.show()

tree_mean = final_test_summary_df.loc[
    final_test_summary_df["model"] == "Tree direct ML", "mean_error_m"
].iloc[0]
tdoa_mean = final_test_summary_df.loc[
    final_test_summary_df["model"] == "TDOA", "mean_error_m"
].iloc[0]

if tree_mean < tdoa_mean:
    print("Tree direct ML beats TDOA on this held-out test split.")
else:
    print(
        "Tree direct ML does not beat TDOA on this held-out test split. Report that honestly and inspect harder scenarios rather than deepening the network immediately."
    )


## 7. Optional: run the repo evaluator

If the repo is available in the Colab runtime, this step will run the existing evaluation script using the exported `ml_predictions.parquet` file.


In [ ]:
EVALUATOR_SCRIPT = "/ML Pipeline/evaluate_positioning_performance.py"

if EVALUATOR_SCRIPT is None:
    print("Evaluator script not found in this runtime.")
    print("If needed, clone the repo in Colab or update EVALUATOR_SCRIPT manually.")
else:
    print(f"Running evaluator: {EVALUATOR_SCRIPT}")
    !python "{EVALUATOR_SCRIPT}" --data-dir "{DATA_DIR}" --predictions-path "{PREDICTIONS_PATH}"
